# Kushmanda — Sati's own language model

Model 4 of the nine: a GPT-style decoder written from scratch and trained from random initialisation on permissively licensed security text (MITRE ATT&CK + CAPEC, CISA KEV, this repository's writing). Its own BPE tokenizer is compared with GPT-2's.

Use a **GPU** runtime (Kaggle: Settings → Accelerator → GPU; needs a phone-verified account).

In [ ]:
import os
import subprocess
import sys
# Run on Kaggle or Colab. Clones this repository and installs the two packages.
if not os.path.exists("PASHUPATASTRA"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/preetraval45/PASHUPATASTRA.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "PASHUPATASTRA/packages/core", "-e", "PASHUPATASTRA/packages/sati", "matplotlib"], check=True)
sys.path[:0] = ["PASHUPATASTRA/packages/core", "PASHUPATASTRA/packages/sati"]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'tiktoken'], check=True)

In [ ]:
import collections
from pathlib import Path
from sati.lm import corpus
docs = corpus.build(Path("cache/corpus"), Path("PASHUPATASTRA"))
train, test = corpus.split(docs)
print(len(docs), collections.Counter(x["source"] for x in docs))
print(len(train), "train /", len(test), "held out")
print(docs[0]["text"][:400])

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "sati.lm.train", "--out", "results/kushmanda", "--root", "../..", "--cache", "../../cache/corpus"], check=True, cwd="PASHUPATASTRA/packages/sati")

In [ ]:
import json
import pandas as pd
r = json.load(open("PASHUPATASTRA/packages/sati/results/kushmanda/results.json"))
print(r["held_out"], r["tokenizer"])
pd.DataFrame(r["curve"]).plot(x="step", y=["train_loss", "held_out_loss"], figsize=(8, 3))
for s in r["samples"]:
    print("\n>>", s["prompt"], s["completion"])